# N031 · 前缀聚合与区间查询

**目标：** 明确半开区间并区分可逆和不可逆聚合。

**先修：** N017, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前缀和/异或；加法逆元；前后缀乘积；prefix min不能直接相减。

**配套讲解来源：** [同名逐章意见](../../comment/031_prefix_sums_xor_products.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两类问题。

**问题 A（区间和反复查询）**：给你一个固定不变的数组，之后会问你很多次"下标 l 到 r 这一段的和是多少"。如果每次都现场把那一段加一遍，查询一多就慢了；我们希望先花一点时间做预处理，之后每次查询都一步出答案。具体例子：数组 `[1,2,3]`，查询 `query(1,3)`，答案是 `5`——因为半开区间 [1,3) 包含下标 1、2 两个元素，即 `2+3=5`。注意本章采用**半开区间**约定：`query(left,right)` 查的是下标 left, left+1, …, right−1，不含 right 本身。再如 `query(2,2)` 查的是空区间，答案是 `0`——这是半开约定最自然的好处之一。

**问题 B（除自身乘积）**：给你数组 nums，要返回新数组 answer，其中 `answer[i]` 等于 nums 中除 `nums[i]` 之外所有元素的乘积，而且**不许用除法**。具体例子：输入 `[1,2,3,4]`，输出 `[24,12,8,6]`——比如 answer[0] = 2×3×4 = 24，answer[2] = 1×2×4 = 8。为什么不许除法？因为数组里可能有 0（比如 `[0,2,3]` 应输出 `[6,0,0]`），一旦有 0，"总乘积除以自己"这条路就整个废掉。

本章标题里的"聚合"指的是求和、求异或、求乘积这类把一段数合成一个数的运算；核心问题是：**哪些聚合可以"先存前缀、再用两个前缀相减/相消还出任意区间"，哪些不行**。判据一句话：运算可逆（有逆元）就行，不可逆就另想办法。

## 2. 基础知识：先读懂这些词

- **前缀和（prefix sum）**：从开头累加到每个位置的累计值。`P[0]=0`，`P[i+1]=P[i]+a[i]`。也就是 P[i] 表示数组前 i 个元素的和。有了它，任意区间 [l,r) 的和就是 `P[r]-P[l]`，一步算完。
- **半开区间 [l,r)**：包含 l、不包含 r 的下标区间。好处有三个：空区间是 [i,i)（和自然为 0）、整段是 [0,n)（不用写 n−1）、区间长度恰好是 r−l。这三种情况在半开约定下公式完全统一，不用分类讨论。
- **可逆聚合与不可逆聚合**：求和是"可逆"的——知道 P[l] 和 P[r]，用减法就能把中间那段单独还出来，因为加法的逆运算是减法。异或（xor）也可逆，而且它自己的逆运算就是它自己（`a xor b xor b = a`，一个数异或两次同一个值就还原）。但 min/max 不可逆：知道"前 l 个的最小值"和"前 r 个的最小值"，你没法用它们算出"第 l 到 r−1 个的最小值"——最小值一旦在压缩时被"吸收"就再也分不开了。乘法理论上可逆（除法），但有 0 时 0 没有逆元（不能除以 0），所以实际中要绕开。判断口诀：想用"前缀相减还区间"，先问聚合运算有没有逆元。
- **加法逆元**：一个数和它的逆元相加等于 0（比如 5 和 −5）。前缀和能用减法还原区间，本质就是因为每个数都有加法逆元。乘法的逆元是倒数，但 0 没有倒数——这正是"除自身乘积"不能硬除的原因。
- **前后缀乘积**：`left[i]` 表示 i 左边所有数的乘积，`right[i]` 表示 i 右边所有数的乘积。那么 `answer[i] = left[i] × right[i]`，全程只用乘法，零元素也照样工作。
  - 直观画面：想象把数组里第 i 个位置"挖个坑"，左边所有数的乘积从左边倒进来，右边所有数的乘积从右边倒进来，坑里装的就是除自身以外的总乘积。
- **PrefixSum 类**：本章把"建前缀表 + 查询"封装成一个类，构造时 O(n) 建表，之后每次 `query` 都是 O(1)。这是"一次预处理、多次查询"的经典工程形态。类名里没有"半开"二字，但接口契约（下一节的代码）明确按半开区间执行，读代码时别按双闭理解。

## 3. 思路推导：从小例子开始

先推问题 A：

- **Step 1**：定义 P[0]=0（空前缀的和是 0，这个 0 是让"从下标 0 开始的区间"也能统一用 P[r]−P[l] 的关键）。
- **Step 2**：从左到右递推 P[i+1] = P[i] + a[i]，得到长 n+1 的前缀表。
- **Step 3**：查询 [l,r) 时返回 P[r] − P[l]。两段前缀共享 [0,l) 部分，相减时公共部分正好抵消，剩下 [l,r) 的和。
  - 抵消的直觉：P[r] 是"前 r 个的和"，P[l] 是"前 l 个的和"，后者是前者的一段前缀。大前缀减去自己的小前缀，剩下的就是夹在中间那段。

手算演示（`a=[1,2,3,4]`，这也是 “运行示例”部分 小实例的数据）：

| i | a[i] | P[i] | 含义 |
|---|---|---|---|
| 0 | — | 0 | 前 0 个元素的和 |
| 1 | 1 | 1 | 前 1 个：1 |
| 2 | 2 | 3 | 前 2 个：1+2 |
| 3 | 3 | 6 | 前 3 个：1+2+3 |
| 4 | 4 | 10 | 前 4 个：1+2+3+4 |

这张表就是 notebook “运行示例”部分 打印的第一张表（列：位置 i、P[i]、前缀区间 `[0,i)`）。用它查 [1,3)：P[3]−P[1] = 6−1 = 5，正是 2+3。查 [2,2)（空区间）：P[2]−P[2]=0。

再推问题 B：

- **Step 1**：从左到右扫，维护滚动乘积 `product`（初始 1）。走到 i 时，`answer[i] 先记成 product`——此刻 product 恰好是 i **严格左边**所有数的乘积。
- **Step 2**：记完再把 `nums[i]` 乘进 product，继续右走。
- **Step 3**：再从右到左扫一遍，同样维护滚动乘积，把 i **严格右边**的乘积乘到 answer[i] 上。左边 × 右边 = 除自身以外的全部。

手算演示（`nums=[1,2,3,4]`）：

- 第一遍（从左到右，先记左乘积再更新）：

| i | 进入时 product（左乘积） | answer[i] 记为 | 离开时 product |
|---|---|---|---|
| 0 | 1 | 1 | 1 |
| 1 | 1 | 1 | 2 |
| 2 | 2 | 2 | 6 |
| 3 | 6 | 6 | 24 |

- 第二遍（从右到左，先乘右乘积再更新）：

| i | 进入时 product（右乘积） | answer[i] 更新 | 离开时 product |
|---|---|---|---|
| 3 | 1 | 6×1=6 | 4 |
| 2 | 4 | 2×4=8 | 12 |
| 1 | 12 | 1×12=12 | 24 |
| 0 | 24 | 1×24=24 | 24（无人再用） |

- 两遍合起来 answer = [24,12,8,6]，与期望一致。这两张表对应的数据也正是 “运行示例”部分 打印的第二张表（输入 [1,2,3,4] 和 [0,2,3] 两组的除自身乘积结果），你可以先手算再运行核对。

## 4. 核心代码：PrefixSum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class PrefixSum:

    def __init__(self, nums):
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def query(self, left, right):
        if not 0 <= left <= right < len(self.prefix):
            raise IndexError('invalid half-open interval')
        return self.prefix[right] - self.prefix[left]
```

### 逐段读懂代码

先看 `PrefixSum` 类，对应实现如下：

```python
class PrefixSum:

    def __init__(self, nums):
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def query(self, left, right):
        if not 0 <= left <= right < len(self.prefix):
            raise IndexError('invalid half-open interval')
        return self.prefix[right] - self.prefix[left]
```

- `self.prefix=[0]`：前缀表从 [0] 起步，这个 0 就是 P[0]（空前缀）。有了它表长是 n+1，`query(l,n)` 查"从 l 到结尾"也不会越界。
- `for x in nums: self.prefix.append(self.prefix[-1]+x)`：`self.prefix[-1]` 是 Python 取列表最后一个元素的写法，即"上一个前缀和"；新前缀 = 旧前缀 + 当前元素。一行完成递推，不需要额外变量。
  - 换成显式下标写法是 `self.prefix.append(self.prefix[i]+nums[i])`，两者完全等价，紧凑版省掉了循环变量 i。
- `if not 0<=left<=right<len(self.prefix): raise IndexError(...)`：这是防御式检查。`0<=left<=right<len(prefix)` 连着写是 Python 的链式比较，意思是四个条件同时成立才合法。left>right（比如 query(3,1)）或 right 超过 n 都是非法半开区间，直接抛异常让调用方立刻发现 bug，而不是默默返回一个错数。
- `return self.prefix[right]-self.prefix[left]`：两个前缀相减，公共的 [0,left) 部分自动抵消。
  - 用 [1,2,3] 的表（prefix=[0,1,3,6]）核对 query(1,3)：6−1=5，正是 2+3；query(0,3)：6−0=6，整段和。左端取 0 时 P[0]=0 不起作用，这就是"空前缀预置为 0"带来的便利。

再看 `product_except_self`，对应实现如下：

```python
def product_except_self(nums):
    answer = [1] * len(nums)
    product = 1
    for i, x in enumerate(nums):
        answer[i] = product
        product *= x
    product = 1
    for i in range(len(nums) - 1, -1, -1):
        answer[i] *= product
        product *= nums[i]
    return answer
```

- `answer=[1]*len(nums); product=1`：answer 预填 1（乘法单位元），product 是滚动乘积，也从 1 开始。
- `for i,x in enumerate(nums): answer[i]=product; product*=x`：注意两条语句的**顺序**——先记再乘。走到 i 时 product 里还是 i 左边全部的乘积，把它存进 answer[i]，然后才把 x 乘进去。如果顺序反了，answer[i] 就会错误地包含 nums[i] 自己。
- `product=1`：重置滚动乘积，准备反向扫描。
- `for i in range(len(nums)-1,-1,-1):`：从最后一个下标走到 0。`range(n-1,-1,-1)` 的三个参数是"起点、终点（不含）、步长"，即 n−1, n−2, …, 0。
- `answer[i]*=product; product*=nums[i]`：同样先乘再更新。此刻 product 是 i 右边全部的乘积，乘上第一遍存的左乘积，answer[i] 就齐了。
- 整个函数只用 answer 一个输出数组加两个标量，没有除法，也没有额外辅助数组。

## 5. 分段实现：按顺序运行

**本章接口：** `PrefixSum`、`product_except_self(nums)`

### PrefixSum

In [1]:
class PrefixSum:

    def __init__(self, nums):
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def query(self, left, right):
        if not 0 <= left <= right < len(self.prefix):
            raise IndexError('invalid half-open interval')
        return self.prefix[right] - self.prefix[left]

### product_except_self

In [2]:
def product_except_self(nums):
    answer = [1] * len(nums)
    product = 1
    for i, x in enumerate(nums):
        answer[i] = product
        product *= x
    product = 1
    for i in range(len(nums) - 1, -1, -1):
        answer[i] *= product
        product *= nums[i]
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,4]; p=PrefixSum(a)
show_table(['位置i','P[i]','前缀区间'],[(i,v,f'[0,{i})') for i,v in enumerate(p.prefix)])
show_table(['输入','除自身乘积'],[(a,product_except_self(a)),([0,2,3],product_except_self([0,2,3]))])

位置i,P[i],前缀区间
0,0,"[0,0)"
1,1,"[0,1)"
2,3,"[0,2)"
3,6,"[0,3)"
4,10,"[0,4)"


输入,除自身乘积
"[1, 2, 3, 4]","[24, 12, 8, 6]"
"[0, 2, 3]","[6, 0, 0]"


## 7. 正确性、适用条件与复杂度

两前缀相减取消共同的[0,l)部分。乘积首遍记录严格左侧，逆遍乘严格右侧，两部分互不重叠且恰好覆盖除当前位置之外的全部索引。

**代价：** 前缀构建 O(n)、查询 O(1)、空间 O(n)；乘积 O(n) 时间、O(1) 辅助空间（不计输出）。

### 展开理解

**为什么对（区间和）**：P[r] 是 [0,r) 的和，P[l] 是 [0,l) 的和，两者相减时 [0,l) 这段公共部分被原样减掉，剩下的正好是 [l,r) 的和。这个抵消不需要任何额外条件，因为加法里每个数都能被减掉。查询前做范围检查，保证下标永远落在表内。

**为什么对（除自身乘积）**：对每个位置 i，第一遍循环存进去的是"下标严格小于 i 的全部元素之积"，第二遍乘上去的是"下标严格大于 i 的全部元素之积"。这两部分没有任何重叠，合起来又恰好覆盖了除 i 之外的所有下标——既不缺谁也不多算谁。因为有 0 也不用怕：全程只做乘法，0 参与乘法没有任何歧义。

**复杂度**：PrefixSum 构建跑一遍数组，O(n) 次加法；每次查询就是一次减法，O(1)。空间上多存一张长 n+1 的表，O(n)。具体数字：数组十万长、十万次查询时，暴力做法最坏是 10^5 × 10^5 = 百亿次加法，跑不完；前缀和方案是十万次建表加十万次减法，二十万次操作，瞬间完成。product_except_self 是两趟线性扫描，时间 O(n)、辅助空间 O(1)（answer 按惯例算输出，不计入辅助空间）。

一个值得记住的对比：前缀表是"空间换时间"的典型——多花一张 O(n) 的表，把每次查询从 O(n) 压到 O(1)。如果你的场景只有一次查询，那直接暴力加一遍反而更快（O(n) 一次 vs O(n) 建表加 O(1) 查询，一样是 O(n) 但系数更小）；前缀和的收益在"查询次数多"时才体现出来。想通这一点，你以后就能自己判断什么时候该上这类预处理结构。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解释：

- `p=PrefixSum([1,2,3]); assert p.query(1,3)==5 and p.query(2,2)==0`：正常值加空区间测试。query(1,3) 查 [1,3) 即 2+3=5，验证主公式；query(2,2) 查空区间 [2,2)，和必须是 0，验证半开约定下空区间的行为。
- `assert product_except_self([1,2,3,4])==[24,12,8,6]`：正常值测试。LeetCode 238 样例，四项乘积我们在第三节手算过。
- `assert product_except_self([0,2,3])==[6,0,0]`：特殊值测试。数组含一个 0：除 0 位置外乘积都是 0，0 位置自己是 2×3=6。这条专门卡"用总乘积硬除"的错误做法——总乘积是 0，除法全线崩溃。顺带这也是 “运行示例”部分 第二张表打印的两组数据之一（输入 [0,2,3] 对输出 [6,0,0]），表格与断言互相印证。
- `assert product_except_self([0,2,0])==[0,0,0]`：特殊值测试。两个 0：任何位置除自身外都还剩一个 0，所以全 0。再进一步排除"用除法 + 特判一个零"的侥幸实现。
- `assert product_except_self([])==[]`：边界值测试。空数组进、空数组出，检查循环边界没有把空输入搞出异常或错误长度。
- 最后的双层循环：对建好的 p（数组 [1,2,3]），枚举所有 0≤l≤r≤3 的半开区间，断言 `p.query(l,r)==sum([1,2,3][l:r])`，即和 Python 切片求和的朴素定义完全一致。拆开看这 10 个区间（l 从 0 到 3，r 从 l 到 3）：
  - 空区间：query(0,0)、query(1,1)、query(2,2)、query(3,3)，全部应得 0（半开区间左闭右开，l==r 时切出来是空）。
  - 单元素：query(0,1)=1、query(1,2)=2、query(2,3)=3。
  - 多元素：query(0,2)=3、query(1,3)=5、query(0,3)=6。
  - 注意 Python 的切片 `a[l:r]` 天生就是半开语义，所以参考答案和我们用的是同一套约定，这条测试本质上是在验证"类没有把半开搞成双闭或双开"。

In [4]:
p = PrefixSum([1, 2, 3])

assert p.query(1, 3) == 5 and p.query(2, 2) == 0

assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]

assert product_except_self([0, 2, 3]) == [6, 0, 0]

assert product_except_self([0, 2, 0]) == [0, 0, 0]

assert product_except_self([]) == []

for l in range(4):
    for r in range(l, 4):
        assert p.query(l, r) == sum([1, 2, 3][l:r])

print('N031: 所有本章断言通过')

N031: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 写出前缀异或查询。

**练习 2：** 构造前缀最小值无法恢复任意区间最小值的反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（写出前缀异或查询）**：提示：把 PrefixSum 里的 `+` 全换成 `^`（Python 的异或运算符），查询把 `-` 换成 `^`。可行的原因：异或的逆运算就是它自己，`P[r]^P[l]` 会把 [0,l) 的公共部分"异或掉"（x^x=0，0^y=y）。手算例子：a=[1,2,3]，前缀异或表是 [0,1,3,0]（0^1=1，1^2=3，3^3=0）；查询 [1,3) 用前缀算是 P[3]^P[1]=0^1=1，直接算元素是 2^3=1，两者一致，公式成立。你写完后再用 [0,2) 等区间多对拍几个。
- **练习 2（构造前缀最小值无法恢复区间最小值的反例）**：提示：min 没有逆运算，减法在 min 语义下无意义。取 a=[3,1,2]：前缀最小表 M=[3,1,1]（M[i] 是前 i 个的最小）。区间 [1,3) 的真实最小值是 1，但你从 M[3]=1、M[1]=3 这两个数出发，做任何算术都凑不出有说服力的"1 减 3"式结论；再换 a=[3,1,2] 与 a=[5,1,9] 对比，两者 M[1]、M[3] 相关的组合值可以相同而区间最小不同，说明信息已经在前缀压缩时丢失（1 被记下后，3 是谁再也找不回来）。由此理解"可逆聚合才能前缀差分"这句话。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
class PrefixSum:

    def __init__(self, nums):
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)

    def query(self, left, right):
        if not 0 <= left <= right < len(self.prefix):
            raise IndexError('invalid half-open interval')
        return self.prefix[right] - self.prefix[left]

def product_except_self(nums):
    answer = [1] * len(nums)
    product = 1
    for i, x in enumerate(nums):
        answer[i] = product
        product *= x
    product = 1
    for i in range(len(nums) - 1, -1, -1):
        answer[i] *= product
        product *= nums[i]
    return answer

# 示例与回归测试
p = PrefixSum([1, 2, 3])

assert p.query(1, 3) == 5 and p.query(2, 2) == 0

assert product_except_self([1, 2, 3, 4]) == [24, 12, 8, 6]

assert product_except_self([0, 2, 3]) == [6, 0, 0]

assert product_except_self([0, 2, 0]) == [0, 0, 0]

assert product_except_self([]) == []

for l in range(4):
    for r in range(l, 4):
        assert p.query(l, r) == sum([1, 2, 3][l:r])

print('N031: 所有本章断言通过')

N031: 所有本章断言通过


## 11. 代表题与迁移

- **303. Range Sum Query - Immutable**：这题就是 `PrefixSum` 类的原题（构造一次、多次查询区间和），练的是"半开前缀表 + 两前缀相减"。这题输入用闭区间下标（left、right 都含），映射到本章接口就是 `query(left, right+1)`，加一那步是两套约定之间唯一的换算。
- **238. Product of Array Except Self**：这题就是 `product_except_self` 的原题，练的是"不许除法时用左右两次扫描拼乘积"，核心收获是识别"0 让乘法不可逆"这件事。题目还追问 O(1) 辅助空间的写法，本章的滚动乘积版正好达标（输出数组不计入）。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。